# 05 — Redes convolucionales (CNN)

**Pregunta (H1):** ¿clasifica mejor las prendas una red que comparte pesos en el espacio?

La Dense de 04 mira cada píxel por separado. Una CNN pasa pequeños filtros por toda la foto: el mismo patrón se reconoce esté donde esté. Comparamos las dos con las mismas fotos RGB64, tres semillas y un número de parámetros parecido (±10 %). La CNN tiene un máximo de 10 épocas por semilla y la Dense de 04, de 30. Esta diferencia limita la atribución de los resultados a la arquitectura.

Después, en una segunda fase, cambiamos **una sola cosa** de la CNN: añadimos BatchNorm.

> **Cómo leer este cuaderno.** Cada sección explica qué se hace y cómo leer el resultado. Las figuras están en inglés; las tablas y explicaciones, en castellano. Test no se abre en ningún momento.

| Palabra | Qué significa aquí |
|---|---|
| **Convolución** | Multiplicar un filtro pequeño (3×3) por cada vecindario de la imagen y sumar |
| **Filtro** | Los 3×3 pesos (por canal) que se aprenden; son los mismos en toda la imagen |
| **Mapa de características** | La imagen de respuestas que produce un filtro |
| **Max-pooling** | Quedarse con el valor mayor de cada bloque 2×2: reduce el tamaño a la mitad |
| **Campo receptivo** | Zona de la foto original que influye en una neurona; crece con cada bloque |
| **BatchNorm** | Normalizar las respuestas de cada canal dentro de cada batch durante el entrenamiento |

In [ ]:
from pathlib import Path
import sys, json, inspect, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/data/local_dataset.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.models.cnn import CNNClassifier, multiply_adds
from src.models.dense import DenseClassifier, parameter_count
from src.training.dense_experiment import load_context, prepare_arrays, normalized_tensors, grouped_intervals
from src.training.saved_experiments import load_saved_experiment
from src.training.cnn_experiment import load_cnn_context, load_saved_phase, run_phase, model_from_config
from src.visualization.style import apply_style, format_number as number, format_percent as percent
from src.visualization.tables import display_es
from src.visualization.dense import show_compact, show_code, run_with_progress
from src.visualization import cnn as figures
apply_style()

# Banderas: Run All solo carga resultados. Entrenar exige activarlas expresamente.
ENTRENAR_CNN = False         # fase h1: CNN sin BatchNorm, tres semillas
ENTRENAR_BATCHNORM = False   # fase batchnorm: solo después de terminar h1

context = load_cnn_context(ROOT)
config = context['config']
names, rows = context['names'], context['rows']
eda = ROOT/'results/metrics/03_eda/dataset_provenance.json'
if not eda.exists() or json.loads(eda.read_text())['manifest_sha256'] != context['manifest_sha256']:
    raise ValueError('Ejecuta primero 03 con la publicación vigente de 02.')
# La referencia Dense de 04 se carga sin entrenar y debe usar la misma publicación.
dense_context = load_context(ROOT)
dense_summary, DENSE = load_saved_experiment(dense_context)
saved_h1 = None if ENTRENAR_CNN else load_saved_phase(context, 'h1')
if saved_h1 is None and not ENTRENAR_CNN:
    raise FileNotFoundError('No hay resultados guardados de la CNN (fase h1) para esta publicación. '
                            'Para entrenar, poner ENTRENAR_CNN = True: no se entrena por faltar una caché.')
torch.set_num_threads(config['training']['threads'])
torch.use_deterministic_algorithms(True)
display_es(pd.DataFrame({
    'Decisión': ['PyTorch', 'Dispositivo', 'Hilos CPU', 'Semillas', 'Batch', 'Máximo épocas', 'Paciencia',
                 'Learning rate', 'Pesos de clase', 'Aumentos'],
    'Valor': [torch.__version__, 'CPU', config['training']['threads'], '42, 123 y 2026', 32, config['training']['max_epochs'], 5, '0,001', 'No', 'No'],
}))

## 1. Mismos datos que la Dense

Usamos la misma publicación de 02 y **exactamente las mismas fotos** de train y validation que 04. Así cualquier diferencia viene del modelo, no de los datos.

**Cómo leerlo:** si una comprobación falla, el cuaderno se detiene; no se compara nada con datos distintos.

In [ ]:
display(Markdown(f"**Publicación:** `{context['run'].name}`. Manifiesto verificado por SHA-256."))
display_es(pd.DataFrame({'Partición': ['train', 'validation', 'test'],
                         'Imágenes': [context['split_counts'][s] for s in ['train', 'validation', 'test']],
                         'Uso aquí': ['Aprender pesos y normalización', 'Parada, checkpoint y análisis',
                                      'Reservado; no se abren fotografías']}))
dense_predictions = pd.read_csv(DENSE/'baseline_seed42_predictions.csv.gz', dtype={'article_id': 'string', 'group_id': 'string'})
assert dense_predictions.article_id.tolist() == rows['validation'].article_id.tolist()
assert dense_summary['identity']['manifest_sha256'] == context['manifest_sha256']
missing_validation = sorted(set(names) - set(rows['validation'].label))
if missing_validation:
    display(Markdown(f"**Clases sin fotos en validation:** {', '.join(missing_validation)}. Se aprenden en train, "
                     f"pero no se pueden evaluar. Macro-F1 promedia las {len(names)} clases, como en 03 y 04, así que cuentan con F1 = 0."))
print('Controles superados: misma publicación y mismas fotos de validation que la Dense de 04.')

## 2. Qué hace una convolución

Tomamos una foto de train y le pasamos un filtro **fijo** de 3×3 que detecta bordes verticales. El filtro se desliza por toda la imagen con **los mismos nueve pesos**.

**Cómo leer la figura:** a la izquierda, la foto; en el centro, el filtro; a la derecha, su mapa de respuestas. Los colores intensos marcan bordes verticales (claro→oscuro u oscuro→claro). La CNN no usa este filtro: aprende los suyos.

In [ ]:
prepared_at = time.perf_counter()
raw, normalization = prepare_arrays(context)
display(Markdown(f"Preparación/verificación: **{number(time.perf_counter() - prepared_at, 1)} s** (misma caché RGB64 que 04)."))
show_compact(figures.convolution_figure(context, raw), width=760)

## 3. La arquitectura

Cuatro bloques **convolución 3×3 → ReLU → max-pooling 2×2**. Cada bloque reduce la imagen a la mitad (64 → 32 → 16 → 8 → 4) y tiene más filtros (16 → 32 → 64 → 128). Al final, una Dense de 400 neuronas y la salida de clases.

**Cómo leer la tabla:** *campo receptivo* es el tamaño de la zona de la foto que puede influir en una neurona de ese bloque. La segunda tabla compara el tamaño con la Dense: los **parámetros** son parecidos, pero la CNN hace **muchas más operaciones**, porque reutiliza sus filtros en cada posición.

In [ ]:
example = model_from_config(config, {'batchnorm': False})
show_code(inspect.getsource(CNNClassifier), 'Ver el código de la CNN')
layers, shape, receptive, jump = [], (1, 3, 64, 64), 1, 1
x = torch.zeros(shape)
for index, block in enumerate(example.features, 1):
    with torch.no_grad():
        x = block(x)
    receptive += 2 * jump          # convolución 3×3
    receptive += jump; jump *= 2   # max-pooling 2×2
    layers.append({'Capa': f'Bloque {index}: conv 3×3 + ReLU + pool', 'Salida': str(tuple(x.shape[1:])),
                   'Parámetros': sum(p.numel() for p in block.parameters()), 'Campo receptivo (px)': receptive})
for module in example.classifier:
    if isinstance(module, torch.nn.Linear):
        layers.append({'Capa': f'Dense {module.in_features} → {module.out_features}', 'Salida': str((module.out_features,)),
                       'Parámetros': sum(p.numel() for p in module.parameters()), 'Campo receptivo (px)': 64})
display_es(pd.DataFrame(layers))
dense_model = DenseClassifier(classes=len(names))
cnn_parameters, dense_parameters = parameter_count(example), parameter_count(dense_model)
size = pd.DataFrame({'Modelo': ['Dense (04)', 'CNN (05)'],
                     'Parámetros': [dense_parameters, cnn_parameters],
                     'Multiplicaciones-sumas por imagen': [multiply_adds(dense_model), multiply_adds(example)]})
display_es(size)
difference = cnn_parameters / dense_parameters - 1
assert abs(difference) <= config['parameter_matching']['tolerance']
display(Markdown(f"Parámetros de la CNN respecto a la Dense: **{percent(difference)}** (límite ±10 %). "
                 f"Operaciones: **{number(size.iloc[1, 2] / size.iloc[0, 2], 1)} veces** las de la Dense."))

## 4. Entrenar o cargar

En el modo normal el cuaderno **carga** las tres ejecuciones guardadas. Si no existen, se detiene con un aviso. Para entrenar por primera vez se pone `ENTRENAR_CNN = True` en la celda de configuración.

Antes de entrenar se mide **una época real** (que se descarta) para estimar el tiempo de las tres semillas. Si la estimación supera el presupuesto de la fase (110 minutos), el entrenamiento no empieza y no se quita ninguna foto automáticamente.

**Cómo leer las tablas:** *época elegida* es la de mejor macro-F1 en validation; *épocas realizadas* incluye la parada temprana.

In [ ]:
tensors = normalized_tensors(raw, normalization)
if ENTRENAR_CNN:
    h1_summary, H1 = run_with_progress(lambda: run_phase(context, tensors, normalization, 'h1'), ROOT/'tmp/05_training_h1.log')
else:
    h1_summary, H1 = saved_h1
cnn_records = pd.DataFrame(h1_summary['runs'])
assert h1_summary['test_images_opened'] == 0 and h1_summary['test_predictions_generated'] == 0
assert set(cnn_records.seed) == set(config['training']['seeds'])
feasibility = h1_summary['cost_feasibility']
display_es(pd.DataFrame({'Medición': ['Época piloto (s)', 'Estimación conservadora de la fase (min)', 'Presupuesto (min)'],
                         'Valor': [feasibility['pilot_epoch_seconds'], feasibility['conservative_upper_seconds']/60,
                                   feasibility['budget_seconds']/60]}))
display(Markdown(f"**Fase h1 completada:** `{h1_summary['experiment_id']}`. Resultados en `results/metrics/05_cnn/runs/{H1.name}`."))
display_es(cnn_records[['variant', 'seed', 'train_macro_f1', 'validation_macro_f1', 'validation_accuracy']])
display_es(cnn_records[['variant', 'seed', 'selected_epoch', 'epochs_run', 'train_seconds']])

## 5. Curvas de aprendizaje

**Morado: train. Naranja: validation.** Cada estilo de línea es una semilla.

**Cómo leerlas:** en la pérdida, bajar es mejorar; en macro-F1, subir. Si train sigue mejorando y validation se estanca o empeora, la red empieza a memorizar train.

In [ ]:
show_compact(figures.curves_figure(context, H1), width=850)
train_f1, val_f1 = cnn_records.train_macro_f1.mean(), cnn_records.validation_macro_f1.mean()
display(Markdown(f"Macro-F1 medio en los checkpoints: train **{number(train_f1, 3)}**, validation **{number(val_f1, 3)}**; "
                 f"diferencia **{number(train_f1 - val_f1, 3)}**. Describe la generalización en esta muestra; no identifica su causa."))

## 6. H1 · CNN frente a Dense

Comparamos **semilla a semilla**: la CNN y la Dense con la misma semilla ven los mismos datos. La diferencia emparejada evita confundir el efecto del modelo con la suerte de una semilla.

El intervalo de confianza se calcula remuestreando **productos** de validation (no fotos sueltas) con las predicciones de las dos redes a la vez.

**Cómo leerlo:** si el intervalo de la diferencia no incluye el 0, la ventaja es consistente en esta muestra de validation. No es todavía un resultado de test.

In [ ]:
dense_records = pd.DataFrame(dense_summary['runs']).query("variant == 'baseline'").set_index('seed')
paired = pd.DataFrame({'Dense (04)': dense_records.validation_macro_f1,
                       'CNN (05)': cnn_records.set_index('seed').validation_macro_f1})
paired.index.name = 'Semilla'
paired['Diferencia CNN − Dense'] = paired['CNN (05)'] - paired['Dense (04)']
display_es(paired)
y = rows['validation'].label_id.astype(int).to_numpy()
groups = rows['validation'].group_id
intervals = {}
for seed in config['training']['seeds']:
    cnn_pred = pd.read_csv(H1/f'cnn_seed{seed}_predictions.csv.gz').predicted_label_id.to_numpy()
    dense_pred = pd.read_csv(DENSE/f'baseline_seed{seed}_predictions.csv.gz').predicted_label_id.to_numpy()
    ci = grouped_intervals(y, cnn_pred, groups, len(names), draws=config['bootstrap']['draws'],
                           seed=config['bootstrap']['seed'], other=dense_pred)
    intervals[seed] = ci[0][1], ci[1][1]
display_es(pd.DataFrame(intervals, index=['IC95 inferior', 'IC95 superior']).T.rename_axis('Semilla'))
show_compact(figures.paired_figure(context, paired['Dense (04)'], paired['CNN (05)'], ['Dense (04)', 'CNN (05)'],
             '03_h1_pairs', 'H1: does spatial weight sharing help?',
             'Source: validation macro-F1 of the selected checkpoint; same seed and data. CNN allows 10 epochs and Dense allows 30. '
             'Parameters matched within 10 %; operations are not.'), width=760)
delta = paired['Diferencia CNN − Dense']
display(Markdown(f"La CNN supera a la Dense en **{int((delta > 0).sum())} de 3 semillas**; diferencia media "
                 f"**{number(delta.mean(), 3)}** de macro-F1 (desviación típica {number(delta.std(), 3)})."))
paired.to_csv(H1/'analysis_h1_pairs.csv')

## 7. Dónde acierta y dónde falla

Usamos la semilla 42 de cada red como ilustración.

**Cómo leerlo:** las barras comparan el F1 de cada clase; las matrices muestran qué clases se confunden (filas = clase real); la galería enseña los errores con más confianza, elegidos automáticamente.

In [ ]:
from sklearn.metrics import confusion_matrix
cnn_predictions = pd.read_csv(H1/'cnn_seed42_predictions.csv.gz', dtype={'article_id': 'string', 'group_id': 'string'})
cnn_report = json.loads((H1/'cnn_seed42_per_class.json').read_text(encoding='utf-8'))
dense_report = json.loads((DENSE/'baseline_seed42_per_class.json').read_text(encoding='utf-8'))
per_class = pd.DataFrame({'F1 Dense': [dense_report[n]['f1-score'] for n in names],
                          'F1 CNN': [cnn_report[n]['f1-score'] for n in names],
                          'Imágenes': [cnn_report[n]['support'] for n in names]}, index=pd.Index(names, name='Clase'))
per_class['Diferencia'] = per_class['F1 CNN'] - per_class['F1 Dense']
display_es(per_class.sort_values('Diferencia'))
show_compact(figures.per_class_figure(context, dense_report, cnn_report), width=900)
cm = confusion_matrix(y, cnn_predictions.predicted_label_id, labels=np.arange(len(names)))
show_compact(figures.confusion_figure(context, cm), width=900)
show_compact(figures.errors_figure(context, cnn_predictions), width=720)

## 8. Qué aprende la CNN

Mostramos los 16 filtros de la primera capa y las respuestas de cada bloque a una foto de train.

**Cómo leerlo:** los filtros son pesos, no explicaciones. Los primeros bloques suelen responder a bordes y colores; los últimos son pequeños (4×4) y más abstractos. Que un mapa se ilumine no demuestra que esa zona cause la predicción.

In [ ]:
reference = cnn_records.query('seed == 42').iloc[0]
model, checkpoint = figures.checkpoint_model(context, reference, model_from_config, {'batchnorm': False})
if model is None:
    display(Markdown(f"**Pesos no disponibles en este equipo:** `{Path(str(reference.checkpoint).replace(chr(92), '/')).name}`. "
                     "Las figuras de filtros y mapas quedan pendientes; el resto no depende de ellas."))
else:
    show_compact(figures.filters_figure(context, model), width=850)
    show_compact(figures.feature_maps_figure(context, model, tensors, raw), width=850)

## 9. Segunda fase · BatchNorm

Misma CNN, mismas semillas y mismos datos; **solo** se añade BatchNorm después de cada convolución. Se entrena después de guardar la fase h1, con `ENTRENAR_BATCHNORM = True`.

**Cómo leerlo:** si no hay resultados guardados, la sección lo indica y el resto del cuaderno sigue siendo válido.

In [ ]:
saved_bn = None if ENTRENAR_BATCHNORM else load_saved_phase(context, 'batchnorm')
if ENTRENAR_BATCHNORM:
    saved_bn = run_with_progress(lambda: run_phase(context, tensors, normalization, 'batchnorm'), ROOT/'tmp/05_training_batchnorm.log')
if saved_bn is None:
    bn_records = None
    display(Markdown('**Fase BatchNorm pendiente.** Activar `ENTRENAR_BATCHNORM = True` cuando se decida entrenarla.'))
else:
    bn_summary, BN = saved_bn
    bn_records = pd.DataFrame(bn_summary['runs'])
    assert bn_summary['test_images_opened'] == 0
    show_compact(figures.curves_figure(context, BN, 'cnn_batchnorm', 'How the CNN learns with BatchNorm'), width=850)
    bn_pairs = pd.DataFrame({'CNN': cnn_records.set_index('seed').validation_macro_f1,
                             'CNN + BatchNorm': bn_records.set_index('seed').validation_macro_f1})
    bn_pairs.index.name = 'Semilla'
    bn_pairs['Diferencia'] = bn_pairs['CNN + BatchNorm'] - bn_pairs['CNN']
    display_es(bn_pairs)
    display_es(bn_records[['seed', 'selected_epoch', 'epochs_run', 'train_seconds']])
    show_compact(figures.paired_figure(context, bn_pairs['CNN'], bn_pairs['CNN + BatchNorm'], ['CNN', 'CNN + BatchNorm'],
                 '09_batchnorm_pairs', 'A single change: adding BatchNorm',
                 'Source: validation macro-F1 of the selected checkpoint; same CNN, seeds, data and budget.'), width=760)
    bn_pairs.to_csv(BN/'analysis_batchnorm_pairs.csv')

## 10. Coste y procedencia

**Cómo leerlo:** el tiempo de entrenamiento incluye la evaluación de cada época; no incluye preparar las fotos. La memoria es la de todo el proceso de Python. La procedencia permite saber qué datos, código y versiones produjeron estos resultados.

In [ ]:
cost = figures.cost_frame(pd.concat([cnn_records, bn_records]) if bn_records is not None else cnn_records)
display_es(cost)
display(Markdown(f"**Entrenamiento total de la fase h1:** {number(cnn_records.train_seconds.sum() / 60, 1)} minutos. "
                 f"Hardware: `{h1_summary['hardware']['processor']}`, CPU con {h1_summary['hardware']['threads_used']} hilos."))
display_es(pd.DataFrame({'Elemento': ['Dataset', 'Configuración CNN', 'Código del modelo', 'PyTorch', 'Referencia Dense'],
                         'Referencia': [h1_summary['dataset_run'], h1_summary['identity']['sources']['config/cnn_baseline_v1.json'],
                                        h1_summary['identity']['sources']['src/models/cnn.py'],
                                        h1_summary['identity']['versions']['torch'], dense_summary['experiment_id']]}))

## 11. Conclusiones

Las cifras se calculan a partir de los resultados anteriores; no se rellenan con resultados supuestos.

In [ ]:
dense_mean = dense_records.validation_macro_f1.mean()
lines = [f"- **CNN:** macro-F1 medio de validation **{number(val_f1, 3)} ± {number(cnn_records.validation_macro_f1.std(), 3)}** "
         f"entre tres semillas; accuracy media **{percent(cnn_records.validation_accuracy.mean())}**.",
         f"- **H1:** frente a la Dense ({number(dense_mean, 3)}), diferencia media emparejada **{number(delta.mean(), 3)}**; "
         f"la CNN gana en **{int((delta > 0).sum())}/3 semillas**, con un número de parámetros parecido y "
         f"{number(size.iloc[1, 2] / size.iloc[0, 2], 1)} veces más operaciones.",
         f"- **Generalización:** separación train − validation de **{number(train_f1 - val_f1, 3)}** en macro-F1."]
if bn_records is not None:
    lines.append(f"- **BatchNorm:** diferencia media **{number(bn_pairs['Diferencia'].mean(), 3)}**; mejora en "
                 f"**{int((bn_pairs['Diferencia'] > 0).sum())}/3 semillas**.")
else:
    lines.append('- **BatchNorm:** fase pendiente de entrenar.')
display(Markdown('\n'.join(lines)))

### Límites

- Son resultados de **validation**, que también se usa para elegir la época: no sustituyen la evaluación final en test.
- Igualar parámetros no iguala operaciones ni tiempo; se informa de ambos.
- Una sola arquitectura y un solo presupuesto: no hemos buscado anchuras, profundidades ni learning rates.
- Aumentos (H2) y pesos de clase (H4) se estudian en 09. Atención sobre la CNN queda como extensión.
- La evaluación describe las prendas seleccionadas tras los controles, no todo el catálogo.

**Material de clase:** `PRDL_MLLB_CNN.pptx`, `PRDL_MLLB_RestNets_to_NeuralODE.pptx`, `MLLB_CNN_MNIST_in_Keras.ipynb`, `Gemini_MNIST_Pytorch_featuremaps.ipynb`.